# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Explore record sets and their fields
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
print("Record set @id and name:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '<no name>')}")

# Preview fields in each record set
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']} fields:")
    fields = rs.get('field', [])
    # 'field' can be dict or list
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        fname = field.get('name', '<no name>')
        f_id = field.get('@id', '<no id>')
        f_type = field.get('dataType', '<no type>')
        print(f"  - {f_id} (name: {fname}, type: {f_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    # Use @id as key
    dataframes[rec_id] = pd.DataFrame(records)

# Show columns of the first non-empty dataframe
for rec_id in record_set_ids:
    df = dataframes[rec_id]
    if not df.empty:
        print(f"\nFirst non-empty record set: {rec_id}")
        print("Columns:", df.columns.tolist())
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You'll need to select a numeric field and a grouping field from the field names above (replace `<numeric_field>` and `<group_field>` with their `@id` or DataFrame column name).

In [ ]:
# Replace these with fields found in previous cells
# EXAMPLE (adjust to actual column names in your dataset!):
example_record_set_id = rec_id  # The record set id printed above
df = dataframes[example_record_set_id]

# Try to infer a numeric field for demonstration
numeric_candidate = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_candidate = col
        break
if not numeric_candidate:
    print("No numeric field found. Please specify the correct column name.")
else:
    print(f"Using numeric field: {numeric_candidate}")
    threshold = df[numeric_candidate].quantile(0.75) if not df[numeric_candidate].isnull().all() else 0
    filtered_df = df[df[numeric_candidate] > threshold]
    print(f"Filtered records with {numeric_candidate} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_candidate}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
    print(f"Normalized {numeric_candidate} for filtered records:")
    print(filtered_df[[numeric_candidate, norm_col]].head())

    # Try to find a categorical/group field (non-numeric, with few unique values)
    group_candidate = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < 10:
            group_candidate = col
            break
    if group_candidate:
        print(f"Grouping by: {group_candidate}")
        grouped_df = filtered_df.groupby(group_candidate)[numeric_candidate].mean().to_frame()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field (if available)
if numeric_candidate:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_candidate].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_candidate}")
    plt.xlabel(numeric_candidate)
    plt.show()

    # If grouping field present, make a boxplot
    if group_candidate:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_candidate, y=numeric_candidate, data=df)
        plt.title(f"{numeric_candidate} by {group_candidate}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR² dataset using its Croissant schema, explored its metadata, record sets, and fields.
- We extracted tabular data and demonstrated basic filtering and normalization on a selected numeric field.
- We grouped and visualized the data to reveal its distribution and sample relationships.

Further, more in-depth analyses can be performed based on the scientific use case!